In [ ]:
# =========================================================
# TFM - Detección de fraude en seguros de automóvil
# Notebook limpio
# =========================================================

# =========================
# 1. Librerías
# =========================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)


# =========================
# 2. Carga del dataset
# =========================
# Opción A: CSV local
df = pd.read_csv("carclaims.csv")

# Opción B: KaggleHub
# import kagglehub
# path = kagglehub.dataset_download("khusheekapoor/vehicle-insurance-fraud-detection")
# df = pd.read_csv(os.path.join(path, "carclaims.csv"))

print(df.shape)
df.head()

In [ ]:
# =========================
# 3. Exploración inicial
# =========================
print(df["FraudFound"].value_counts())
print(df["FraudFound"].value_counts(normalize=True) * 100)

df["FraudFound"].value_counts().plot(kind="bar")
plt.title("Distribución de FraudFound")
plt.xlabel("FraudFound")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# =========================
# 4. Limpieza básica
# =========================
df_clean = df.copy()

# Eliminar variable identificativa
df_clean = df_clean.drop(columns=["PolicyNumber"])

# Eliminar Age por valores anómalos y redundancia
df_clean = df_clean.drop(columns=["Age"])

# Recodificar categorías anómalas
df_clean["DayOfWeekClaimed"] = df_clean["DayOfWeekClaimed"].replace("0", "Unknown")
df_clean["MonthClaimed"] = df_clean["MonthClaimed"].replace("0", "Unknown")

# Recodificar variable objetivo
df_clean["FraudFound"] = (
    df_clean["FraudFound"]
    .astype(str)
    .str.strip()
    .map({"No": 0, "Yes": 1})
)

print(df_clean["FraudFound"].value_counts(dropna=False))

In [ ]:
# =========================
# 5. Recodificación ordinal
# =========================
df_model = df_clean.copy()

map_age_policyholder = {
    "16 to 17": 1,
    "18 to 20": 2,
    "21 to 25": 3,
    "26 to 30": 4,
    "31 to 35": 5,
    "36 to 40": 6,
    "41 to 50": 7,
    "51 to 65": 8,
    "over 65": 9
}

map_age_vehicle = {
    "new": 0,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "more than 7": 8
}

map_vehicle_price = {
    "less than 20,000": 1,
    "20,000 to 29,000": 2,
    "30,000 to 39,000": 3,
    "40,000 to 59,000": 4,
    "60,000 to 69,000": 5,
    "more than 69,000": 6
}

map_days_policy_accident = {
    "none": 0,
    "1 to 7": 1,
    "8 to 15": 2,
    "15 to 30": 3,
    "more than 30": 4
}

map_days_policy_claim = {
    "none": 0,
    "8 to 15": 1,
    "15 to 30": 2,
    "more than 30": 3
}

map_past_claims = {
    "none": 0,
    "1": 1,
    "2 to 4": 2,
    "more than 4": 3
}

df_model["AgeOfPolicyHolder"] = df_model["AgeOfPolicyHolder"].map(map_age_policyholder)
df_model["AgeOfVehicle"] = df_model["AgeOfVehicle"].map(map_age_vehicle)
df_model["VehiclePrice"] = df_model["VehiclePrice"].astype(str).str.strip().map(map_vehicle_price)
df_model["Days:Policy-Accident"] = df_model["Days:Policy-Accident"].map(map_days_policy_accident)
df_model["Days:Policy-Claim"] = df_model["Days:Policy-Claim"].map(map_days_policy_claim)
df_model["PastNumberOfClaims"] = df_model["PastNumberOfClaims"].map(map_past_claims)

ordinal_cols = [
    "AgeOfPolicyHolder",
    "AgeOfVehicle",
    "VehiclePrice",
    "Days:Policy-Accident",
    "Days:Policy-Claim",
    "PastNumberOfClaims"
]

print(df_model[ordinal_cols].isnull().sum())

In [ ]:
# =========================
# 6. Agrupación de categorías poco frecuentes
# =========================
frecuencias_make = df_model["Make"].value_counts()
marcas_frecuentes = frecuencias_make[frecuencias_make >= 100].index
df_model["Make"] = df_model["Make"].apply(lambda x: x if x in marcas_frecuentes else "Other")

df_model["Make"].value_counts()

In [ ]:
# =========================
# 7. Construcción de la matriz final
# =========================
X = df_model.drop(columns=["FraudFound"])
y = df_model["FraudFound"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
print(X_encoded.shape)

In [ ]:
# =========================
# 8. División entrenamiento / prueba
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# =========================
# 9. Escalado para regresión logística
# =========================
X_train_num = X_train[num_cols].copy()
X_test_num = X_test[num_cols].copy()

dummy_cols = [col for col in X_train.columns if col not in num_cols]
X_train_dummy = X_train[dummy_cols].copy()
X_test_dummy = X_test[dummy_cols].copy()

scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num),
    columns=num_cols,
    index=X_train_num.index
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num),
    columns=num_cols,
    index=X_test_num.index
)

X_train_logit = pd.concat([X_train_num_scaled, X_train_dummy], axis=1).sort_index(axis=1)
X_test_logit = pd.concat([X_test_num_scaled, X_test_dummy], axis=1).sort_index(axis=1)

In [ ]:
# =========================
# 10. Regresión logística
# =========================
modelo_logit = LogisticRegression(random_state=42, max_iter=1000)
modelo_logit.fit(X_train_logit, y_train)

y_pred_logit = modelo_logit.predict(X_test_logit)

print("Regresión logística")
print("Accuracy:", accuracy_score(y_test, y_pred_logit))
print(confusion_matrix(y_test, y_pred_logit))
print(classification_report(y_test, y_pred_logit, zero_division=0))

In [ ]:
# =========================
# 11. Regresión logística balanceada
# =========================
modelo_logit_bal = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)
modelo_logit_bal.fit(X_train_logit, y_train)

y_pred_logit_bal = modelo_logit_bal.predict(X_test_logit)

print("Regresión logística balanceada")
print("Accuracy:", accuracy_score(y_test, y_pred_logit_bal))
print(confusion_matrix(y_test, y_pred_logit_bal))
print(classification_report(y_test, y_pred_logit_bal, zero_division=0))

In [ ]:
# =========================
# 12. Árbol de decisión básico
# =========================
modelo_tree = DecisionTreeClassifier(random_state=42)
modelo_tree.fit(X_train, y_train)

y_pred_tree = modelo_tree.predict(X_test)

print("Árbol de decisión básico")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree, zero_division=0))

In [ ]:
# =========================
# 13. Árbol de decisión balanceado
# =========================
modelo_tree_bal = DecisionTreeClassifier(
    random_state=42,
    max_depth=6,
    min_samples_leaf=20,
    class_weight="balanced"
)
modelo_tree_bal.fit(X_train, y_train)

y_pred_tree_bal = modelo_tree_bal.predict(X_test)

print("Árbol de decisión balanceado")
print("Accuracy:", accuracy_score(y_test, y_pred_tree_bal))
print(confusion_matrix(y_test, y_pred_tree_bal))
print(classification_report(y_test, y_pred_tree_bal, zero_division=0))

In [ ]:
# =========================
# 14. Función auxiliar para evaluar umbrales
# =========================
def evaluar_umbral(y_true, y_prob, umbral):
    y_pred = (y_prob >= umbral).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "umbral": umbral,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_fraude": report["1"]["precision"],
        "recall_fraude": report["1"]["recall"],
        "f1_fraude": report["1"]["f1-score"]
    }

umbrales = [0.50, 0.30, 0.20, 0.10, 0.08, 0.05, 0.03]

In [ ]:
# =========================
# 15. Random Forest
# =========================
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
modelo_rf.fit(X_train, y_train)

y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

resultados_rf = pd.DataFrame([evaluar_umbral(y_test, y_prob_rf, u) for u in umbrales])
resultados_rf = resultados_rf.sort_values(by="f1_fraude", ascending=False).reset_index(drop=True)

resultados_rf

In [ ]:
# Mejor versión de Random Forest
mejor_umbral_rf = resultados_rf.loc[0, "umbral"]
y_pred_rf = (y_prob_rf >= mejor_umbral_rf).astype(int)

print("Random Forest - mejor umbral:", mejor_umbral_rf)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, zero_division=0))

In [ ]:
# =========================
# 16. XGBoost
# =========================
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()
scale_pos_weight = negativos / positivos

modelo_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

modelo_xgb.fit(X_train, y_train)

y_prob_xgb = modelo_xgb.predict_proba(X_test)[:, 1]

resultados_xgb = pd.DataFrame([evaluar_umbral(y_test, y_prob_xgb, u) for u in umbrales])
resultados_xgb = resultados_xgb.sort_values(by="f1_fraude", ascending=False).reset_index(drop=True)

resultados_xgb

In [ ]:
# Mejor versión de XGBoost
mejor_umbral_xgb = resultados_xgb.loc[0, "umbral"]
y_pred_xgb = (y_prob_xgb >= mejor_umbral_xgb).astype(int)

print("XGBoost - mejor umbral:", mejor_umbral_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, zero_division=0))

In [ ]:
# =========================
# 17. Curva Precision-Recall y AUC-PR de XGBoost
# =========================
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_prob_xgb)
auc_pr_xgb = average_precision_score(y_test, y_prob_xgb)

print("AUC-PR XGBoost:", auc_pr_xgb)

plt.figure(figsize=(8, 6))
plt.plot(recall_xgb, precision_xgb, label=f"XGBoost (AUC-PR = {auc_pr_xgb:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall - XGBoost")
plt.legend()
plt.show()

In [ ]:
# =========================
# 18. Tabla comparativa final
# =========================
tabla_modelos = pd.DataFrame([
    {
        "Modelo": "Regresión logística",
        "Accuracy": accuracy_score(y_test, y_pred_logit),
        "Precision_fraude": classification_report(y_test, y_pred_logit, output_dict=True, zero_division=0)["1"]["precision"],
        "Recall_fraude": classification_report(y_test, y_pred_logit, output_dict=True, zero_division=0)["1"]["recall"],
        "F1_fraude": classification_report(y_test, y_pred_logit, output_dict=True, zero_division=0)["1"]["f1-score"]
    },
    {
        "Modelo": "Regresión logística balanceada",
        "Accuracy": accuracy_score(y_test, y_pred_logit_bal),
        "Precision_fraude": classification_report(y_test, y_pred_logit_bal, output_dict=True, zero_division=0)["1"]["precision"],
        "Recall_fraude": classification_report(y_test, y_pred_logit_bal, output_dict=True, zero_division=0)["1"]["recall"],
        "F1_fraude": classification_report(y_test, y_pred_logit_bal, output_dict=True, zero_division=0)["1"]["f1-score"]
    },
    {
        "Modelo": "Árbol de decisión básico",
        "Accuracy": accuracy_score(y_test, y_pred_tree),
        "Precision_fraude": classification_report(y_test, y_pred_tree, output_dict=True, zero_division=0)["1"]["precision"],
        "Recall_fraude": classification_report(y_test, y_pred_tree, output_dict=True, zero_division=0)["1"]["recall"],
        "F1_fraude": classification_report(y_test, y_pred_tree, output_dict=True, zero_division=0)["1"]["f1-score"]
    },
    {
        "Modelo": "Árbol de decisión balanceado",
        "Accuracy": accuracy_score(y_test, y_pred_tree_bal),
        "Precision_fraude": classification_report(y_test, y_pred_tree_bal, output_dict=True, zero_division=0)["1"]["precision"],
        "Recall_fraude": classification_report(y_test, y_pred_tree_bal, output_dict=True, zero_division=0)["1"]["recall"],
        "F1_fraude": classification_report(y_test, y_pred_tree_bal, output_dict=True, zero_division=0)["1"]["f1-score"]
    },
    {
        "Modelo": "Random Forest (mejor versión)",
        "Accuracy": resultados_rf.loc[0, "accuracy"],
        "Precision_fraude": resultados_rf.loc[0, "precision_fraude"],
        "Recall_fraude": resultados_rf.loc[0, "recall_fraude"],
        "F1_fraude": resultados_rf.loc[0, "f1_fraude"]
    },
    {
        "Modelo": "XGBoost",
        "Accuracy": resultados_xgb.loc[0, "accuracy"],
        "Precision_fraude": resultados_xgb.loc[0, "precision_fraude"],
        "Recall_fraude": resultados_xgb.loc[0, "recall_fraude"],
        "F1_fraude": resultados_xgb.loc[0, "f1_fraude"]
    }
])

tabla_modelos = tabla_modelos.sort_values(by="F1_fraude", ascending=False).reset_index(drop=True)
tabla_modelos.round(3)

In [ ]:
# =========================
# 19. Importancia de variables del mejor modelo
# =========================
importancias_xgb = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo_xgb.feature_importances_
}).sort_values(by="Importancia", ascending=False)

importancias_xgb.head(20)